In [1]:
import pandas as pd
import numpy as np

In [25]:
df = pd.read_csv("marketing_campaign_data.csv")

In [26]:
print(f"Data has {df.shape[0]} rows and {df.shape[1]} columns")

Data has 2020 rows and 12 columns


In [27]:
df

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA


In [28]:
df.columns

Index([' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel',
       'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks',
       'Campaign_Tag'],
      dtype='object')

In [29]:
#Removing extra spaces from column names
df.columns = df.columns.str.strip().str.lower()

In [30]:
df.columns

Index(['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel',
       'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks',
       'campaign_tag'],
      dtype='object')

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   campaign_id    2020 non-null   object 
 1   campaign_name  2020 non-null   object 
 2   start_date     2020 non-null   object 
 3   end_date       2020 non-null   object 
 4   channel        1919 non-null   object 
 5   impressions    2020 non-null   int64  
 6   clicks         2020 non-null   int64  
 7   spend          2020 non-null   object 
 8   conversions    1820 non-null   float64
 9   active         2020 non-null   object 
 10  clicks         40 non-null     float64
 11  campaign_tag   2020 non-null   object 
dtypes: float64(2), int64(2), object(8)
memory usage: 189.5+ KB


In [32]:
#Few values in spend column has $ sign: will remove $ sign and change type to numeric
dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print("Before Cleaning:")
print(df.loc[dirty_spend_mask, ['spend']].head(3))

#remvoing $ sign
df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]','', regex=True)
df['spend'] = pd.to_numeric(df['spend'])


print("After Cleaning:")
print(df.loc[dirty_spend_mask, ['spend']].head(3))



Before Cleaning:
       spend
0    $102.82
21   $2428.4
22  $4726.22
After Cleaning:
      spend
0    102.82
21  2428.40
22  4726.22


In [33]:
#Channel has few inconsistant categories which means same category written in different formats
print("Before Cleaning:")
print(df['channel'].unique())

cleaned_names = {
    'Facebok': 'Facebook',
    'Tik_Tok' : 'TikTok',
    'E-mail' : 'Email',
    'Gogle' : 'Google Ads',
    'Insta_gram' : 'Instagram',
    'NA' : np.nan
}

df['channel'] = df['channel'].replace(cleaned_names)

print("After Cleaning:")

print(df['channel'].unique())

Before Cleaning:
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
After Cleaning:
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan]


In [34]:
#Fixing boolean values
print("Before Cleaning:")
print(df['active'].unique())

bool_fix = {
    'Y' : True,
    'Yes' : True,
    '1' : True,
    'No' : False,
    '0' : False
}

df['active'] = df['active'].map(bool_fix).fillna(False).astype(bool)

print("After Cleaning:")
print(df['active'].unique())

Before Cleaning:
['Y' '0' 'No' 'True' 'Yes' '1' 'False']
After Cleaning:
[ True False]


C:\Users\asifv\AppData\Local\Temp\ipykernel_8608\1864809815.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['active'] = df['active'].map(bool_fix).fillna(False).astype(bool)


In [35]:
#Date column cleaning
print(df['start_date'].dtype)

df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print("After Fix:")
print(df['start_date'].dtype)

object
After Fix:
datetime64[ns]


C:\Users\asifv\AppData\Local\Temp\ipykernel_8608\984624368.py:5: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')


In [36]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,clicks,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,NaN,FA


In [37]:
#dropping duplicate column
df = df.loc[:, ~df.columns.duplicated()]

In [38]:
# Logical fix (Impression must be higher than clicks)
impossible_mask = df['clicks'] > df['impressions']
print(df.loc[impossible_mask, ['campaign_id', 'impressions', 'clicks']].head(3))

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


In [39]:
#Logical check (No start_date after end_date)
date_mask = df['end_date'] < df['start_date']
print(df.loc[date_mask, ['campaign_id', 'start_date', 'end_date']].head(3))
df.loc[date_mask, 'end_date'] = df.loc[date_mask, 'end_date'] + pd.Timedelta(days=30)
print("After fix:")
print(df.loc[date_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27
After fix:
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-31
54   CMP-00055 2023-09-01 2023-09-26
71   CMP-00072 2023-02-01 2023-02-26


In [42]:
# Detecting outliers
Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

IQR = Q3 - Q1

upper_boundary = Q3 + (3 * IQR)

outlier_mask = df['spend'] > upper_boundary
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

print("After fix:")
df.loc[outlier_mask, 'spend'] = upper_boundary
print(df.loc[outlier_mask, ['campaign_id', 'spend']].head(3))

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
After fix:
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375
